# HERTA evaluation and visualization

Edit the configuration cell, then run the cells in order. Inputs are predicted stripe NPZ files and matching Hi-C z-score dictionaries. The examples use 10 kb bins. For S2D predictions, add `_s2d` after `{cell_type}` in `PREDICTION_PATTERN`. Use trusted pickle inputs.


In [ ]:
from pathlib import Path
import os
import sys
import pickle
import warnings
import itertools

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import logging

logging.getLogger("fontTools").setLevel(logging.WARNING)
logging.getLogger("fontTools.subset").setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
from scipy import sparse
from scipy.sparse import coo_matrix
from scipy.stats import pearsonr, spearmanr
from sklearn import metrics
from mpl_toolkits.axes_grid1 import make_axes_locatable

warnings.filterwarnings("ignore")

# Fix old scipy sparse pickle module paths
import scipy.sparse as _sp
for _mod in ["_csr", "_csc", "_coo", "_bsr", "_dia", "_lil", "_dok"]:
    sys.modules.setdefault(f"scipy.sparse.{_mod}", _sp)


import math
from itertools import chain
from typing import Union
from contextlib import suppress
from scipy import stats
import scipy.sparse as sp


## Configuration

Set your sample, chromosome list and paths below. `VIS_START`, `PLOT_WIN`, `SHOW_LO` and `SHOW_HI` are bin indices for the contact-map view. Distance-stratified labels and contact-map display defaults are set for 10 kb bins; adjust those plotting settings when using other resolutions.


In [ ]:
# Edit these values for your inputs.
CELL_TYPE = "sample"
TEST_CHROMS = [1]
RESOLUTIONS = [10000]
PRED_DIR = Path("/path/to/predictions")
GT_DIR = Path("/path/to/data/hic")
OUTPUT_DIR = Path("/path/to/evaluation_output")
PREDICTION_PATTERN = "prediction_{cell_type}_{resolution_kb}kb_chr{chrom}.npz"

# Contact-map view; the example uses 10 kb bins.
VIS_CHROM = 1
VIS_START = 0
PLOT_WIN = 700
SHOW_LO = 250
SHOW_HI = 450

RESOLUTION_CONFIG = {
    5000: {
        "label": "5kb",
        "resolution_kb": 5,
        "pred_len": 400,
        "offset": 400,
        "front_crop": 400,
        "hic_suffix": "5kb",
    },
    10000: {
        "label": "10kb",
        "resolution_kb": 10,
        "pred_len": 200,
        "offset": 200,
        "front_crop": 200,
        "hic_suffix": "10kb",
    },
    25000: {
        "label": "25kb",
        "resolution_kb": 25,
        "pred_len": 80,
        "offset": 80,
        "front_crop": 80,
        "hic_suffix": "25kb",
    },
}
INSULATION_WINDOW = 20
SEGMENT_BINS = 200
AUROC_PERCENTILE = 90
N_BOOTSTRAP = 10000
DIST_BIN_EDGES_10KB = [1, 5, 10, 20, 40, 80, 120, 160, 200]
DIST_LABELS_10KB = [
    "10-50kb", "50-100kb", "100-200kb", "200-400kb",
    "400-800kb", "800k-1.2M", "1.2-1.6M", "1.6-2M",
]
NM_SINGLE_COL_INCH = 89 / 25.4
NM_DOUBLE_COL_INCH = 183 / 25.4
NM_FONTSIZE = 7
DGRAY = "#404040"
COLOR_HERTA = "#E69F00"

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": NM_FONTSIZE,
    "axes.labelsize": NM_FONTSIZE,
    "axes.titlesize": NM_FONTSIZE + 1,
    "xtick.labelsize": NM_FONTSIZE,
    "ytick.labelsize": NM_FONTSIZE,
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none",
    "axes.spines.top": True,
    "axes.spines.right": True,
})

METHOD_ORDER = ["HERTA"]
METHODS_CONFIG = {"HERTA": {"pred_dir": str(PRED_DIR), "pred_format": "npz", "gt_type": "zscore"}}


## Scoring

Reconstruction, Pearson/Spearman correlation, insulation, HiCRep SCC, AUROC/AUPRC and bootstrap helpers are defined together below. Insulation helper author attribution: Laura Gunsalus and Evonne McArthur. HiCRep references are retained in the SCC helper docstrings.


In [ ]:
# Insulation and HiCRep SCC helper functions.
BINS = 448

def remove_missing_points_flat(flat_a, flat_b):
    """
    Remove points in vector where either flat_a OR flat_b are NaN
    Input:
        flat_a: numpy vector
        flat_b: numpy vector
    Returns:
        flat_a: numpy vector with nan positions removed
        flat_b: numpy vector with nan positions removed
    """
    mask = np.logical_or(np.isnan(flat_a), np.isnan(flat_b))
    return flat_a[~mask], flat_b[~mask]

def fill_missing_points_map(map_a, map_b, fill=0):
    """
    Fills points in matrix where either map_a OR map_b are NaN
    Input:
        map_a: numpy array
        map_b: numpy array
        fill: value to fill missing points with (default: 0)
    Returns:
        map_a: numpy array with missing positions filled with specified value
        map_b: numpy array with missing positions filled with specified value
    """
    mask = np.logical_or(np.isnan(map_a), np.isnan(map_b))
    map_a_copy, map_b_copy = np.copy(map_a), np.copy(map_b)
    map_a_copy[mask] = fill
    map_b_copy[mask] = fill
    return map_a_copy, map_b_copy

def spearman_1D(vector_a, vector_b):
    """
    Function to calculate the spearman correlation between two 1D arrays.

    Input:
        vector_a: 1D numpy array of length(n)
        vector_b: 1D numpy array of length(n)
    Returns:
        scalar value
    """

    # For Distance enrichment, sometimes the vectors are not the same length so here we need to make them the same length
#     vector_a = vector_a[:min(len(vector_a), len(vector_b))]
#     vector_b = vector_b[:min(len(vector_a), len(vector_b))]
    
    vector_a, vector_b = remove_missing_points_flat(vector_a, vector_b)

    spearmanr_val, pval = stats.spearmanr(vector_a, vector_b)
    return spearmanr_val

def mse_1D(vector_a, vector_b):
    """
    Function to calculate the mean squared error between two 1D arrays.

    Input:
        vector_a: 1D numpy array of length(n)
        vector_b: 1D numpy array of length(n)

    Returns:
        scalar value
    """

    vector_a, vector_b = remove_missing_points_flat(vector_a, vector_b)

    mse = np.mean(np.square(vector_a - vector_b))
    return mse

def vectorMethodToScalar(method, map_a, map_b, finalCompMetric='all', return_tracks=False):
    """
    Wrapper for the functions that output vectors or "tracks"
    and then compare the outputs of two vectors using MSE, SpearmanR, PearsonR.
    Will default with returning a dictionary with all 3 comparison measures.
    Alternatively you can specify the final comparison measure with finalCompMetric.

    Methods that can use this include: insulation_track, contact_directionality_track, calculate_decay_track,
    triangle_track, eigenvector_track,

    Input:
        method: method to convert matrix to flattened vector track (e.g. insulation_track)
        map_a: n x n numpy array
        map_b: n x n numpy array
        finalCompMetric: either 'all', 'corr', or 'mse'
        return_tracks: boolean if you want to return the 1D array tracks
    Returns:
        if finalCompMetric != 'all', returns a scalar value
        if finalCompMetric == 'all', returns a dictionary of comparisons with all 3
        (spearmanr, and mse)
    """

    map_a_filled, map_b_filled = fill_missing_points_map(map_a, map_b, fill=np.nan)

    a_track = method(map_a_filled)
    b_track = method(map_b_filled)
    
    if len(a_track) != len(b_track):
        a_track = a_track[:min(len(a_track), len(b_track))]
        b_track = b_track[:min(len(a_track), len(b_track))]

    if finalCompMetric == 'all':
        output = {'corr': spearman_1D(a_track, b_track),
                  'mse': mse_1D(a_track, b_track)}
    elif finalCompMetric == 'corr':
        output = spearman_1D(a_track, b_track)
    elif finalCompMetric == 'mse':
        output = mse_1D(a_track, b_track)
    else:
        raise ValueError(
            f"finalCompMetric specified ({finalCompMetric}) is not understood. Use either 'all', 'corr', or 'mse'")

    if return_tracks:
        return output, (a_track, b_track)

    return output

def insulation_track(map, window_size=10, plot=False, ax=None):
    """
    get the insulation profile using a diamond-shaped window-based method, specially it scans along
    the diagonal of a matrix using a W by W diamond-shaped window, calculating the average contact
    frequency within each window. The locations at which the average contact frequency reaches a
    local minimum are identified as candidate TAD boundaries.
    Input:
        map: n x n numpy array
        window_size: size of the diamond-shaped window
        plot: True or False
        ax: if you want to plot on an already specified axis
    Returns:
        array: n x 1 insulation track of map
    """

    insulation_track = []

    # Select the diagonal
    for loc in range(0, len(map)):
        # Ignore if it's an edge pixel
        if loc <= window_size or loc >= len(map) - window_size:
            insulation_track.append(np.nan)
            continue

        # Define focal region
        focal_start = loc - window_size - 1
        focal_end = loc + window_size + 1

        window = map[focal_start:loc - 1, loc + 1:focal_end]
        window_mean = np.nanmean([np.exp(i) for i in list(chain(*window))])

        insulation_track.append(math.log2(window_mean))

    if plot:
        if ax is None:
            fig, ax = plt.subplots()

        ax.plot(insulation_track)
        ax.set_xlim(0, BINS)
        ax.set_ylabel('insulation score')

    return np.array(insulation_track)

def upperDiagCsr(m: sp.coo_matrix, nDiags: int):
    """Convert an input sp.coo_matrix into a sp.csr_matrix where each row in the
    the output corresponds to one diagonal of the upper triangle of the input.
    Args:
        m (sp.coo_matrix): input matrix
        nDiags (int): output diagonals with index in the range [1, nDiags)
        as rows of the output matrix
    Returns: `sp.csr_matrix` whose rows are the diagonals of the input
    """
    row = m.col - m.row
    idx = np.where((row > 0) & (row < nDiags))
    idxRowp1 = row[idx]
    # the diagonal index becomes the row index
    idxRow = idxRowp1 - 1
    # offset in the original diagonal becomes the column index
    idxCol = m.col[idx] - idxRowp1
    ans = sp.csr_matrix((m.data[idx], (idxRow, idxCol)),
                        shape=(nDiags - 1, m.shape[1]), dtype=m.dtype)
    ans.eliminate_zeros()
    return ans

def varVstran(n: Union[int, np.ndarray]):
    """
    Calculate the variance of variance-stabilizing transformed
    (or `vstran()` in the original R implementation) data. The `vstran()` turns
    the input data into ranks, whose variance is only a function of the input
    size:
        ```
        var(1/n, 2/n, ..., n/n) = (1 - 1/(n^2))/12
        ```
    or with Bessel's correction:
        ```
        var(1/n, 2/n, ..., n/n, ddof=1) = (1 + 1.0/n)/12
        ```
    See section "Variance stabilized weights" in reference for more detail:
    https://genome.cshlp.org/content/early/2017/10/06/gr.220640.117
    Args:
        n (Union(int, np.ndarray)): size of the input data
    Returns: `Union(int, np.ndarray)` variance of the ranked input data with Bessel's
    correction
    """
    with suppress(ZeroDivisionError), np.errstate(divide='ignore', invalid='ignore'):
        return np.where(n < 2, np.nan, (1 + 1.0 / n) / 12.0)

def sccByDiag(m1: sp.coo_matrix, m2: sp.coo_matrix, nDiags: int):
    """Compute diagonal-wise hicrep SCC score for the two input matrices up to
    nDiags diagonals
    Args:
        m1 (sp.coo_matrix): input contact matrix 1
        m2 (sp.coo_matrix): input contact matrix 2
        nDiags (int): compute SCC scores for diagonals whose index is in the
        range of [1, nDiags)
    Returns: `float` hicrep SCC scores
    """
    # convert each diagonal to one row of a csr_matrix in order to compute
    # diagonal-wise correlation between m1 and m2
    m1D = upperDiagCsr(m1, nDiags)
    m2D = upperDiagCsr(m2, nDiags)
    nSamplesD = (m1D + m2D).getnnz(axis=1)
    rowSumM1D = m1D.sum(axis=1).A1
    rowSumM2D = m2D.sum(axis=1).A1
    # ignore zero-division warnings because the corresponding elements in the
    # output don't contribute to the SCC scores
    with np.errstate(divide='ignore', invalid='ignore'):
        cov = m1D.multiply(m2D).sum(axis=1).A1 - rowSumM1D * rowSumM2D / nSamplesD
        rhoD = cov / np.sqrt(
            (m1D.power(2).sum(axis=1).A1 - np.square(rowSumM1D) / nSamplesD ) *
            (m2D.power(2).sum(axis=1).A1 - np.square(rowSumM2D) / nSamplesD ))
        wsD = nSamplesD * varVstran(nSamplesD)
        # Convert NaN and Inf resulting from div by 0 to zeros.
        # posinf and neginf added to fix behavior seen in 4DN datasets
        # 4DNFIOQLTI9G and DNFIH7MQHOR at 5kb where inf would be reported
        # as an SCC score
        wsNan2Zero = np.nan_to_num(wsD, copy=True, posinf=0.0, neginf=0.0)
        rhoNan2Zero = np.nan_to_num(rhoD, copy=True, posinf=0.0, neginf=0.0)

    return rhoNan2Zero @ wsNan2Zero / wsNan2Zero.sum()

def get_combined_yhat(y_hat_list, start_ind, end_ind, pred_len, offset=None, avg_stripe=False):
    """Reconstruct a contact matrix from predicted v-stripes."""
    if offset is None:
        offset = pred_len
    y_hat_list_reshaped = np.concatenate(
        [x.reshape(1, -1) for x in y_hat_list[start_ind:end_ind]], axis=0
    )
    chrom_length = y_hat_list_reshaped.shape[0]
    mat = []
    for i in range(chrom_length):
        mat.append(np.insert(
            np.zeros(chrom_length + offset + 1), i,
            np.insert(y_hat_list_reshaped[i], pred_len, 0)
        ))
    summed = pd.DataFrame(np.array(mat)).reindex(
        np.arange(-1 * pred_len, chrom_length, 1)
    ).fillna(0).T.iloc[0:chrom_length + pred_len, 0:chrom_length + pred_len].values

    if avg_stripe:
        summed_T = pd.DataFrame(np.array(mat)).reindex(
            np.arange(-1 * pred_len, chrom_length, 1)
        ).fillna(0).iloc[0:chrom_length + pred_len, 0:chrom_length + pred_len].values
        summed = (summed + summed_T) / 2

    summed = summed[pred_len:-pred_len, pred_len:-pred_len]
    return summed


def load_gt_dict(gt_dir, cell_types, resolutions):
    gt_dict = {}
    for ct in cell_types:
        gt_dict[ct] = {}
        for resolution in resolutions:
            rc = RESOLUTION_CONFIG[resolution]
            gt_file = Path(gt_dir) / f"{ct}_hic_zscore_dict_{rc['hic_suffix']}.p"
            if not gt_file.exists():
                raise FileNotFoundError(f"GT file not found: {gt_file}")
            with open(gt_file, "rb") as f:
                gt_dict[ct][resolution] = pickle.load(f)
            print(f"Loaded GT: {ct} {rc['label']} -> {gt_file.name}")
    return gt_dict


def load_pred_and_gt(method, cell_type, chrom, resolution):
    config = METHODS_CONFIG[method]
    res_cfg = RESOLUTION_CONFIG[resolution]
    pred_len = res_cfg["pred_len"]
    offset = res_cfg["offset"]
    front_crop = res_cfg["front_crop"]
    resolution_kb = res_cfg["resolution_kb"]
    chrom_str = f"chr{chrom}"

    pred_file = str(Path(config["pred_dir"]) / PREDICTION_PATTERN.format(cell_type=cell_type, resolution_kb=resolution_kb, chrom=chrom))
    if not os.path.exists(pred_file):
        return None, None

    y_hat = np.load(pred_file)["arr_0"]
    y_hat = np.expand_dims(y_hat, 1)

    pred_mat = get_combined_yhat(
        y_hat[:, 0, :],
        start_ind=0,
        end_ind=y_hat.shape[0],
        pred_len=pred_len,
        offset=offset,
        avg_stripe=True,
    )

    if cell_type not in GT_DICT or resolution not in GT_DICT[cell_type]:
        return None, None

    gt_dict = GT_DICT[cell_type][resolution]
    if chrom_str not in gt_dict:
        return None, None

    gt_obj = gt_dict[chrom_str]
    gt_mat = gt_obj.toarray() if sparse.issparse(gt_obj) else np.asarray(gt_obj)
    gt_mat = gt_mat[front_crop:, front_crop:].clip(-16, 16)

    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_mat = pred_mat[:n, :n]
    gt_mat = gt_mat[:n, :n]
    return pred_mat, gt_mat


def _zero_first_two_diags(mat):
    n = mat.shape[0]
    for k in range(0, 2):
        np.fill_diagonal(mat[k:, :n - k], 0)
        np.fill_diagonal(mat[:n - k, k:], 0)


def compute_insulation_global(pred_mat, gt_mat, window_size=20):
    """
    Insulation comparison:
    vectorMethodToScalar + insulation_track.
    Returned corr is Spearman correlation.
    """
    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n].copy()
    gt_sub = gt_mat[:n, :n].copy()

    _zero_first_two_diags(pred_sub)
    _zero_first_two_diags(gt_sub)

    results = {"matrix_size": n}
    try:
        insulation_method = lambda m: insulation_track(m, window_size=window_size)
        insu = vectorMethodToScalar(insulation_method, pred_sub, gt_sub)
        results["insulation_corr"] = insu["corr"]
        results["insulation_mse"] = insu["mse"]
    except Exception:
        results["insulation_corr"] = np.nan
        results["insulation_mse"] = np.nan

    return results


def compute_insulation_segmented(pred_mat, gt_mat, segment_bins=200, window_size=20):
    """
    Segmented insulation comparison:
    per-segment vectorMethodToScalar + insulation_track.
    """
    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n]
    gt_sub = gt_mat[:n, :n]
    n_segments = n // segment_bins

    insulation_method = lambda m: insulation_track(m, window_size=window_size)
    segment_corrs = []
    segment_mses = []

    for seg_i in range(n_segments):
        s = seg_i * segment_bins
        e = s + segment_bins
        if e > n:
            break

        pred_seg = pred_sub[s:e, s:e].copy()
        gt_seg = gt_sub[s:e, s:e].copy()

        _zero_first_two_diags(pred_seg)
        _zero_first_two_diags(gt_seg)

        try:
            insu = vectorMethodToScalar(insulation_method, pred_seg, gt_seg)
            corr_val = insu["corr"]
            mse_val = insu["mse"]
            if not np.isnan(corr_val):
                segment_corrs.append(corr_val)
                segment_mses.append(mse_val)
        except Exception:
            continue

    return np.array(segment_corrs), np.array(segment_mses)


def compute_scc(pred_mat, gt_mat, n_diags=200):
    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n]
    gt_sub = gt_mat[:n, :n]

    pf, gf = fill_missing_points_map(pred_sub, gt_sub, fill=0)
    m1 = coo_matrix(pf)
    m2 = coo_matrix(gf)
    return sccByDiag(m1, m2, n_diags)


def compute_per_diagonal_corr(pred_mat, gt_mat, max_diag=200):
    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n]
    gt_sub = gt_mat[:n, :n]

    pearson_scores = []
    spearman_scores = []

    for d in range(1, min(max_diag, n)):
        p_diag = np.diag(pred_sub, d)
        g_diag = np.diag(gt_sub, d)
        valid = np.isfinite(p_diag) & np.isfinite(g_diag)

        if valid.sum() < 20:
            continue

        pv = p_diag[valid]
        gv = g_diag[valid]

        if np.std(pv) == 0 or np.std(gv) == 0:
            continue

        r_p = pearsonr(pv, gv)[0]
        r_s = spearmanr(pv, gv)[0]

        if not np.isnan(r_p):
            pearson_scores.append(r_p)
        if not np.isnan(r_s):
            spearman_scores.append(r_s)

    return {
        "pearson_mean": np.mean(pearson_scores) if pearson_scores else np.nan,
        "spearman_mean": np.mean(spearman_scores) if spearman_scores else np.nan,
        "pearson_per_diag": pearson_scores,
        "spearman_per_diag": spearman_scores,
        "n_valid_diags": len(pearson_scores),
    }


def compute_distance_stratified(pred_mat, gt_mat, bin_edges=None, labels=None):
    if bin_edges is None:
        bin_edges = DIST_BIN_EDGES_10KB
    if labels is None:
        labels = DIST_LABELS_10KB

    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n]
    gt_sub = gt_mat[:n, :n]

    results = []
    for i in range(len(bin_edges) - 1):
        d_lo, d_hi = bin_edges[i], bin_edges[i + 1]

        pred_vals = []
        gt_vals = []
        for d in range(d_lo, min(d_hi, n)):
            p_diag = np.diag(pred_sub, d)
            g_diag = np.diag(gt_sub, d)
            valid = np.isfinite(p_diag) & np.isfinite(g_diag)
            pred_vals.append(p_diag[valid])
            gt_vals.append(g_diag[valid])

        pred_vals = np.concatenate(pred_vals) if pred_vals else np.array([])
        gt_vals = np.concatenate(gt_vals) if gt_vals else np.array([])

        entry = {
            "distance_range": labels[i] if i < len(labels) else f"{d_lo}-{d_hi}",
            "n_values": len(pred_vals),
        }

        if len(pred_vals) >= 20 and np.std(pred_vals) > 0 and np.std(gt_vals) > 0:
            entry["pearson"] = pearsonr(pred_vals, gt_vals)[0]
            entry["spearman"] = spearmanr(pred_vals, gt_vals)[0]
        else:
            entry["pearson"] = np.nan
            entry["spearman"] = np.nan

        results.append(entry)

    return results


def compute_auroc_auprc(pred_mat, gt_mat, resolution_kb, percentile=90):
    band_width = int(2000 / resolution_kb)

    n = min(pred_mat.shape[0], gt_mat.shape[0])
    pred_sub = pred_mat[:n, :n]
    gt_sub = gt_mat[:n, :n]

    max_diagonal_offset = band_width - 1
    min_len = min(
        len(np.diag(gt_sub, max_diagonal_offset)),
        len(np.diag(pred_sub, max_diagonal_offset)),
    )

    if min_len < 100:
        return {
            "auroc": np.nan,
            "auprc": np.nan,
            "n_auroc_diags": 0,
            "auroc_per_diag": [],
            "auprc_per_diag": [],
        }

    bin_true = np.concatenate(
        [np.diag(gt_sub, i)[0:min_len] for i in range(1, band_width)]
    ).reshape(-1, min_len)

    bin_mask = bin_true.sum(0) >= np.percentile(bin_true.sum(0), 1)

    all_zval = np.concatenate(
        [np.diag(gt_sub, i)[0:min_len] for i in range(1, band_width)]
    )
    zval_cutoff = np.percentile(all_zval, percentile)

    roc_scores = []
    prc_scores = []

    for i in range(1, band_width):
        if len(np.diag(gt_sub, i)) < min_len or len(np.diag(pred_sub, i)) < min_len:
            continue

        a = np.diag(gt_sub, i)[0:min_len][bin_mask]
        b = np.diag(pred_sub, i)[0:min_len][bin_mask]
        p = a > zval_cutoff

        if p.sum() == 0 or p.sum() == len(p):
            continue

        try:
            fpr, tpr, _ = metrics.roc_curve(p.astype(int), b)
            precision, recall, _ = metrics.precision_recall_curve(p.astype(int), b)
            roc_scores.append(metrics.auc(fpr, tpr))
            prc_scores.append(metrics.auc(recall, precision))
        except Exception:
            continue

    return {
        "auroc": np.mean(roc_scores) if roc_scores else np.nan,
        "auprc": np.mean(prc_scores) if prc_scores else np.nan,
        "n_auroc_diags": len(roc_scores),
        "auroc_per_diag": roc_scores,
        "auprc_per_diag": prc_scores,
    }


def bootstrap_ci(data, n_bootstrap=10000, ci=0.95, stat_func=np.median):
    rng = np.random.default_rng(42)
    boot_stats = np.array([
        stat_func(rng.choice(data, size=len(data), replace=True))
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - ci) / 2
    lo = np.percentile(boot_stats, alpha * 100)
    hi = np.percentile(boot_stats, (1 - alpha) * 100)
    return stat_func(data), lo, hi


print("Data loading and metric functions are ready.")


def calc_apa_score(apa, f):
    center = apa[f, f]
    corners = np.mean([apa[0, 0], apa[0, -1], apa[-1, 0], apa[-1, -1]])
    return center / corners if corners != 0 else np.nan


## Plotting functions


In [ ]:
def method_style(method):
    if method == "HERTA":
        return COLOR_HERTA, "-", "HERTA"
    return "#666666", "-", method

def plot_per_diag_curves(ax, data, resolution_kb=10, metric_label="Pearson", show_ribbon=False):
    for method, vals in data.items():
        color, ls, _ = method_style(method)

        if isinstance(vals, dict):
            arrays = [np.array(v) for v in vals.values() if v is not None and len(v) > 0]
            if not arrays:
                continue

            max_len = max(len(a) for a in arrays)
            padded = np.full((len(arrays), max_len), np.nan)
            for i, a in enumerate(arrays):
                padded[i, :len(a)] = a

            mean = np.nanmean(padded, axis=0)
            sem = np.nanstd(padded, axis=0) / np.sqrt(np.sum(~np.isnan(padded), axis=0))
        else:
            mean = np.array(vals)
            sem = None

        x = (np.arange(len(mean)) + 1) * resolution_kb
        ax.plot(x, mean, color=color, ls=ls, label=method, linewidth=1.0)

        if show_ribbon and sem is not None:
            ax.fill_between(x, mean - sem, mean + sem, color=color, alpha=0.12)

    ax.set_xlabel("Genomic distance (kb)")
    ax.set_ylabel(metric_label)
    ax.legend(fontsize=NM_FONTSIZE - 1)

def plot_metric_per_diag(cell_type, resolution, metric_key, metric_label, ylim=None):
    res_cfg = RESOLUTION_CONFIG[resolution]
    resolution_kb = res_cfg["resolution_kb"]

    fig, ax = plt.subplots(figsize=(NM_SINGLE_COL_INCH, NM_SINGLE_COL_INCH * 0.75))

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color(DGRAY)
        spine.set_linewidth(0.6)

    diag_data = {}
    for method in METHOD_ORDER:
        diag_data[method] = {
            f"chr{c}": ALL_RESULTS[cell_type][resolution][method][c][metric_key]
            for c in TEST_CHROMS
            if c in ALL_RESULTS[cell_type][resolution][method]
            and metric_key in ALL_RESULTS[cell_type][resolution][method][c]
        }

    plot_per_diag_curves(
        ax,
        diag_data,
        resolution_kb=resolution_kb,
        metric_label=metric_label,
        show_ribbon=False,
    )

    if ylim is not None:
        ax.set_ylim(*ylim)

    ax.set_title(f"{cell_type} {res_cfg['label']}", fontsize=NM_FONTSIZE + 1, color=DGRAY)
    ax.tick_params(colors=DGRAY, width=0.6, length=2)
    fig.tight_layout()

    out = OUTPUT_DIR / f"{cell_type}_{res_cfg['label']}_{metric_key}_curve"
    for fmt in ["svg", "pdf", "png"]:
        fig.savefig(f"{out}.{fmt}", dpi=600, bbox_inches="tight", pad_inches=0.02)

    print(f"saved: {out}.svg/.pdf/.png")
    plt.show()

def pcolormesh_45deg(ax, C, vmax, vmin, cmap="RdBu_r"):
    n = C.shape[0]
    t = np.array([[1, 0.5], [-1, 0.5]])
    A = np.dot(
        np.array([
            (i[1], i[0])
            for i in itertools.product(range(n, -1, -1), range(0, n + 1, 1))
        ]),
        t,
    )
    return ax.pcolormesh(
        A[:, 1].reshape(n + 1, n + 1),
        A[:, 0].reshape(n + 1, n + 1),
        np.flipud(C),
        cmap=cmap,
        vmax=vmax,
        vmin=vmin,
        rasterized=True,
    )


def plot_apa_pair(apa_gt, apa_pred, gt_score, pred_score, title, filename_base):
    fig, axes = plt.subplots(1, 2, figsize=(NM_SINGLE_COL_INCH * 1.35, NM_SINGLE_COL_INCH * 0.72))

    for ax, mat, score, label in [
        (axes[0], apa_gt, gt_score, "Experimental Hi-C"),
        (axes[1], apa_pred, pred_score, "HERTA predicted Hi-C"),
    ]:
        vmin = np.percentile(mat, 5)
        vmax = np.percentile(mat, 95)

        im = ax.imshow(
            mat,
            cmap="Reds",
            vmin=vmin,
            vmax=vmax,
            interpolation="nearest",
            aspect="equal",
        )

        sz = mat.shape[0]
        mid = sz // 2

        ax.set_xticks([0, mid, sz - 1])
        ax.set_xticklabels([f"-{mid}", "0", f"+{mid}"], fontsize=NM_FONTSIZE - 1)
        ax.set_yticks([0, mid, sz - 1])
        ax.set_yticklabels([f"-{mid}", "0", f"+{mid}"], fontsize=NM_FONTSIZE - 1)

        ax.set_title(f"{label}\nAPA = {score:.2f}", fontsize=NM_FONTSIZE)
        ax.set_xlabel("Relative position (bins)", fontsize=NM_FONTSIZE)
        if ax is axes[0]:
            ax.set_ylabel("Relative position (bins)", fontsize=NM_FONTSIZE)

        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=NM_FONTSIZE - 2)

    fig.suptitle(title, fontsize=NM_FONTSIZE + 1, fontweight="bold", y=1.02)
    fig.tight_layout()

    out = OUTPUT_DIR / filename_base
    for fmt in ["svg", "pdf", "png"]:
        fig.savefig(f"{out}.{fmt}", dpi=600, bbox_inches="tight", pad_inches=0.02)

    print(f"saved: {out}.svg/.pdf/.png")
    plt.show()

def plot_hic_pair_2mb(
    cell_type,
    chrom,
    vis_start,
    resolution=10000,
    plot_win=700,
    show_lo=250,
    show_hi=450,
    gt_vmin=-0.5,
    gt_vmax=3.5,
    pred_vmin=-0.5,
    pred_vmax=3.5,
    save=True,
):
    pred_h, gt_raw = load_pred_and_gt("HERTA", cell_type, chrom, resolution)
    if pred_h is None:
        raise FileNotFoundError(f"Missing prediction for {cell_type} chr{chrom} {resolution}")

    gt_sym = gt_raw + gt_raw.T - np.diag(np.diag(gt_raw))

    n = min(pred_h.shape[0], gt_sym.shape[0])
    if vis_start + plot_win > n:
        raise ValueError(
            f"Region out of range: vis_start={vis_start}, plot_win={plot_win}, n={n}"
        )

    mat_gt = np.triu(
        gt_sym[
            vis_start:vis_start + plot_win,
            vis_start:vis_start + plot_win,
        ]
    )
    mat_h = np.triu(
        pred_h[
            vis_start:vis_start + plot_win,
            vis_start:vis_start + plot_win,
        ]
    )

    display_gt = gt_sym[
        vis_start + show_lo:vis_start + show_hi,
        vis_start + show_lo:vis_start + show_hi,
    ]
    display_h = pred_h[
        vis_start + show_lo:vis_start + show_hi,
        vis_start + show_lo:vis_start + show_hi,
    ]

    rows, cols = np.triu_indices(show_hi - show_lo, k=1)
    gv = display_gt[rows, cols]
    hv = display_h[rows, cols]
    valid = np.isfinite(gv) & np.isfinite(hv)
    r_h = pearsonr(gv[valid], hv[valid])[0] if valid.sum() > 20 else np.nan

    front_crop = RESOLUTION_CONFIG[resolution]["front_crop"]

    genome_lo_mb = (vis_start + front_crop + show_lo) * (resolution / 1e6)
    genome_hi_mb = (vis_start + front_crop + show_hi) * (resolution / 1e6)

    heatmap_items = [
        (
            mat_gt,
            "Experimental Hi-C\n(HiC-DC+ Z-score)",
            gt_vmax,
            gt_vmin,
        ),
        (
            mat_h,
            f"HERTA\nPredicted Hi-C\n(r={r_h:.3f})",
            pred_vmax,
            pred_vmin,
        ),
    ]

    fig = plt.figure(figsize=(5, 5.6))
    gs = fig.add_gridspec(
        2,
        1,
        height_ratios=[3, 3],
        hspace=0.08,
    )

    for idx, (mat, label, vmax, vmin) in enumerate(heatmap_items):
        ax = fig.add_subplot(gs[idx])

        img = pcolormesh_45deg(
            ax,
            mat,
            vmax=vmax,
            vmin=vmin,
            cmap="RdBu_r",
        )

        ax.set_ylim(0, show_hi - show_lo)
        ax.set_xlim(show_lo, show_hi)

        ax.set_ylabel(
            label,
            fontsize=7,
            rotation=90,
            ha="center",
            va="center",
            labelpad=24,
        )

        ax.set_yticks([0, 50, 100, 150, 200])
        ax.set_yticklabels(["0", "0.5", "1.0", "1.5", "2.0"], fontsize=5)
        ax.tick_params(axis="y", length=2, width=0.5, pad=1)
        ax.text(
            -0.005,
            1.05,
            "Mb",
            transform=ax.transAxes,
            fontsize=5,
            va="bottom",
            ha="right",
        )

        if idx == 0:
            ax.set_xticks([])
        else:
            xt = np.linspace(show_lo, show_hi, 5)
            ax.set_xticks(xt)
            ax.set_xticklabels(
                [f"{x:.1f}" for x in np.linspace(0, 2, 5)],
                fontsize=6,
            )
            ax.set_xlabel("Window position (Mb)", fontsize=7)

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="1.5%", pad=0.02)
        cb = plt.colorbar(img, cax=cax)
        cb.ax.tick_params(labelsize=4, length=1, width=0.3)

    fig.suptitle(
        f"{cell_type} chr{chrom}: {genome_lo_mb:.2f}–{genome_hi_mb:.2f} Mb",
        fontsize=9,
        fontweight="bold",
        y=0.99,
    )

    fig.tight_layout()

    if save:
        out = OUTPUT_DIR / (
            f"{cell_type}_chr{chrom}_"
            f"{genome_lo_mb:.2f}_{genome_hi_mb:.2f}Mb_hic_map"
        )
        for fmt in ["svg", "pdf", "png"]:
            fig.savefig(
                f"{out}.{fmt}",
                dpi=600,
                bbox_inches="tight",
                pad_inches=0.02,
            )
        print(f"saved: {out}.svg/.pdf/.png")

    plt.show()


## Load inputs


In [ ]:
# Check paths before loading matrices.
missing = []
for resolution in RESOLUTIONS:
    res_cfg = RESOLUTION_CONFIG[resolution]
    gt_file = GT_DIR / f"{CELL_TYPE}_hic_zscore_dict_{res_cfg['hic_suffix']}.p"
    if not gt_file.is_file():
        missing.append(str(gt_file))
    for chrom in TEST_CHROMS:
        pred_file = PRED_DIR / PREDICTION_PATTERN.format(
            cell_type=CELL_TYPE, resolution_kb=res_cfg['resolution_kb'], chrom=chrom)
        if not pred_file.is_file():
            missing.append(str(pred_file))
if missing:
    raise FileNotFoundError("Set the input paths in the configuration cell. Missing:\n" + "\n".join(missing))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GT_DICT = load_gt_dict(GT_DIR, [CELL_TYPE], RESOLUTIONS)


## Evaluate

Export per-chromosome metrics to `OUTPUT_DIR`.


In [ ]:
ALL_RESULTS = {}
ALL_SEG = {}
ALL_DIST = {}

FINAL_METRICS = [
    ("pearson", "Pearson"),
    ("insulation_seg_corr_median", "Insulation"),
    ("auroc", "AUROC"),
    ("auprc", "AUPRC"),
]

for cell_type in [CELL_TYPE]:
    ALL_RESULTS[cell_type] = {}
    ALL_SEG[cell_type] = {}
    ALL_DIST[cell_type] = {}

    for resolution in RESOLUTIONS:
        res_cfg = RESOLUTION_CONFIG[resolution]
        resolution_kb = res_cfg["resolution_kb"]
        band_width = int(2000 / resolution_kb)

        ALL_RESULTS[cell_type][resolution] = {}
        ALL_SEG[cell_type][resolution] = {}
        ALL_DIST[cell_type][resolution] = {}

        print("\n" + "=" * 70)
        print(f"{cell_type} @ {resolution_kb}kb")
        print("=" * 70)

        raw_matrices = {}

        for method in METHOD_ORDER:
            raw_matrices[method] = {}
            for chrom in TEST_CHROMS:
                pred_mat, gt_mat = load_pred_and_gt(method, cell_type, chrom, resolution)
                if pred_mat is not None:
                    raw_matrices[method][chrom] = (pred_mat, gt_mat)

        chrom_unified_size = {}
        for chrom in TEST_CHROMS:
            sizes = []
            for method in METHOD_ORDER:
                if chrom in raw_matrices[method]:
                    p, g = raw_matrices[method][chrom]
                    sizes.append(min(p.shape[0], g.shape[0]))
            if sizes:
                chrom_unified_size[chrom] = min(sizes)

        print(
            "Unified matrix sizes:",
            ", ".join(f"chr{c}={chrom_unified_size[c]}" for c in sorted(chrom_unified_size))
        )

        for method in METHOD_ORDER:
            ALL_RESULTS[cell_type][resolution][method] = {}
            ALL_SEG[cell_type][resolution][method] = {}
            ALL_DIST[cell_type][resolution][method] = {}

            print(f"\n--- {method} ---")

            for chrom in TEST_CHROMS:
                chrom_str = f"chr{chrom}"

                if chrom not in raw_matrices[method] or chrom not in chrom_unified_size:
                    print(f"{chrom_str}: SKIP")
                    continue

                pred_mat, gt_mat = raw_matrices[method][chrom]
                n = chrom_unified_size[chrom]
                pred_mat = pred_mat[:n, :n]
                gt_mat = gt_mat[:n, :n]

                diag_res = compute_per_diagonal_corr(
                    pred_mat,
                    gt_mat,
                    max_diag=band_width,
                )

                ins = compute_insulation_global(
                    pred_mat,
                    gt_mat,
                    window_size=INSULATION_WINDOW,
                )

                try:
                    scc_val = compute_scc(pred_mat, gt_mat, n_diags=min(band_width, n))
                except Exception as e:
                    print(f"{chrom_str} SCC failed: {e}")
                    scc_val = np.nan

                auroc_res = compute_auroc_auprc(
                    pred_mat,
                    gt_mat,
                    resolution_kb=resolution_kb,
                    percentile=AUROC_PERCENTILE,
                )

                seg_corr, seg_mse = compute_insulation_segmented(
                    pred_mat,
                    gt_mat,
                    segment_bins=SEGMENT_BINS,
                    window_size=INSULATION_WINDOW,
                )

                dist_res = compute_distance_stratified(pred_mat, gt_mat)

                result = {
                    **diag_res,
                    **ins,
                    "scc": scc_val,
                    **auroc_res,
                    "matrix_size": n,
                    "insulation_seg_corr_median": np.median(seg_corr) if len(seg_corr) else np.nan,
                    "insulation_seg_mse_median": np.median(seg_mse) if len(seg_mse) else np.nan,
                }

                ALL_RESULTS[cell_type][resolution][method][chrom] = result
                ALL_SEG[cell_type][resolution][method][chrom] = (seg_corr, seg_mse)
                ALL_DIST[cell_type][resolution][method][chrom] = dist_res

                print(
                    f"{chrom_str} ({n}×{n}): "
                    f"Pearson={diag_res['pearson_mean']:.4f}, "
                    f"Insulation={result['insulation_seg_corr_median']:.4f}, "
                    f"AUROC={auroc_res['auroc']:.4f}, "
                    f"AUPRC={auroc_res['auprc']:.4f}"
                )

rows = []
for cell_type in ALL_RESULTS:
    for resolution in ALL_RESULTS[cell_type]:
        for method in ALL_RESULTS[cell_type][resolution]:
            for chrom, r in ALL_RESULTS[cell_type][resolution][method].items():
                rows.append({
                    "cell_type": cell_type,
                    "resolution": RESOLUTION_CONFIG[resolution]["label"],
                    "resolution_bp": resolution,
                    "method": method,
                    "chrom": chrom,
                    "pearson": r["pearson_mean"],
                    "insulation_seg_corr_median": r["insulation_seg_corr_median"],
                    "auroc": r["auroc"],
                    "auprc": r["auprc"],
                })

summary_df = pd.DataFrame(rows)

summary_path = OUTPUT_DIR / f"{CELL_TYPE}_evaluation_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "=" * 80)
print("Final metrics by chromosome")
print("=" * 80)

display(
    summary_df[
        [
            "cell_type",
            "resolution",
            "method",
            "chrom",
            "pearson",
            "insulation_seg_corr_median",
            "auroc",
            "auprc",
        ]
    ]
)

print("\n" + "=" * 80)
print("Chromosome average")
print("=" * 80)

avg_df = (
    summary_df
    .groupby(["cell_type", "resolution", "method"], as_index=False)
    [["pearson", "insulation_seg_corr_median", "auroc", "auprc"]]
    .mean()
)

display(avg_df)

print("saved:", summary_path)


## Distance curves


In [ ]:
for metric_key, metric_label, ylim in [
    ("pearson_per_diag", "Pearson correlation", (0, 0.85)),
    ("spearman_per_diag", "Spearman correlation", (0, 0.85)),
    ("auroc_per_diag", "AUROC", (0, 1.0)),
    ("auprc_per_diag", "AUPRC", (0, 1.0)),
]:
    plot_metric_per_diag(
        cell_type=CELL_TYPE,
        resolution=RESOLUTIONS[0],
        metric_key=metric_key,
        metric_label=metric_label,
        ylim=ylim,
    )


## Contact-map comparison


In [ ]:
VIS_CT = CELL_TYPE

plot_hic_pair_2mb(
    cell_type=VIS_CT,
    chrom=VIS_CHROM,
    vis_start=VIS_START,
    resolution=RESOLUTIONS[0],
    plot_win=PLOT_WIN,
    show_lo=SHOW_LO,
    show_hi=SHOW_HI,
    gt_vmin=-0.5,
    gt_vmax=3.5,
    pred_vmin=-0.5,
    pred_vmax=3.5,
    save=True,
)
